# Allows to keep only major raods 

In [ ]:
import geopandas as gpd

# Load the heavy raw street edges
edges = gpd.read_file('../data/geospatial/street_edges.geojson')

# Define what we consider a "huge road"
major_road_types = [
    'motorway', 'motorway_link', 
    'trunk', 'trunk_link', 
    'primary', 'primary_link', 
    'secondary', 'secondary_link'
]

# Filter down to just the major roads
major_edges = edges[edges['highway'].isin(major_road_types)]

print(f"Reduced from {len(edges)} segments to {len(major_edges)} segments.")

# Save as a new, much smaller working file
major_edges.to_file('../data/geospatial/major_roads.geojson', driver='GeoJSON')


# Download ressources for adaptive capacity 
Everything concerning this is inside the data/geospatial/request.md

# Part 2.1: clip heavy rasters to nairobi boundary
The [HDX](https://data.humdata.org/dataset/highresolutionpopulationdensitymaps-ken) population files for the whole of Kenya are very large.  This clip them to the Nairobi boundary to keep only the subpart that is interesting us.

In [20]:
import os
import sys
import rasterio
from rasterio.mask import mask
import rasterio.warp
from rasterio.enums import Resampling
sys.path.append('../.')
from src.common_helper import *

boundary = load_boundary()

def clip_raster_to_boundary(input_tif, output_tif, boundary_gdf):
    """Clips a large raster to the provided boundary GeoDataFrame."""
    if not os.path.exists(input_tif):
        print(f"File not found: {input_tif}")
        return
        
    print(f"Clipping {input_tif}...")
    with rasterio.open(input_tif) as src:
        # Reproject boundary to match raster CRS
        boundary_crs = boundary_gdf.to_crs(src.crs)
        geoms = [geom for geom in boundary_crs.geometry]
        
        # Mask the raster
        out_image, out_transform = mask(src, geoms, crop=True)
        out_meta = src.meta.copy()
        
        out_meta.update({
            "driver": "GTiff",
            "height": out_image.shape[1],
            "width": out_image.shape[2],
            "transform": out_transform
        })
        
        # Save the clipped raster
        with rasterio.open(output_tif, "w", **out_meta) as dest:
            dest.write(out_image)
    print(f"Saved clipped raster to {output_tif}")

sensitivity_dir = "C:/Users/Surface/Desktop/EPFL/MASTER/MA2_2025_2026/PDS-ETHOS/Nairobi_Heat_Vulnerability_Assessment/data/geospatial/sensitivity"
# clip_raster_to_boundary(
#     os.path.join(sensitivity_dir, 'ken_general_2020.tif'), 
#     os.path.join(sensitivity_dir, 'nairobi_general_pop_2020.tif'), 
#     boundary
# )

# clip_raster_to_boundary(
#     os.path.join(sensitivity_dir, 'ken_elderly_60_plus_2020.tif'), 
#     os.path.join(sensitivity_dir, 'nairobi_elderly_pop_2020.tif'), 
#     boundary
# )

clip_raster_to_boundary(
    os.path.join(sensitivity_dir, 'ken_children_under_five_2020.tif'), 
    os.path.join(sensitivity_dir, 'nairobi_children_pop_2020.tif'), 
    boundary
)


Clipping C:/Users/Surface/Desktop/EPFL/MASTER/MA2_2025_2026/PDS-ETHOS/Nairobi_Heat_Vulnerability_Assessment/data/geospatial/sensitivity\ken_children_under_five_2020.tif...
Saved clipped raster to C:/Users/Surface/Desktop/EPFL/MASTER/MA2_2025_2026/PDS-ETHOS/Nairobi_Heat_Vulnerability_Assessment/data/geospatial/sensitivity\nairobi_children_pop_2020.tif
